# 第112章 GridSearch与随机搜索

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 27 / 34 步：调优、比较、解释并保存模型**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 交叉验证策略  →  **本章任务：** GridSearch与随机搜索  →  **下一步：** 模型比较与基线
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

训练机器学习模型时，很多算法的效果高度依赖超参数（比如 SVM 的 C 和 gamma、随机森林的树数量）。


## 本章目标

学完本章，你将能够：

- **理解**：理解「GridSearch与随机搜索」的核心概念、适用场景与关键口径。
- **操作**：能按本章步骤写出可复现的实现，并读懂输出/结果。
- **迁移**：能用本章方法处理一份新数据，独立完成同类任务并给出结论。


## 核心概念

**背景引入**：训练机器学习模型时，很多算法的效果高度依赖超参数（比如 SVM 的 C 和 gamma、随机森林的树数量）。手动一个个试不仅慢，还容易漏掉好的组合。GridSearch 系统地枚举一组候选组合、随机搜索在大空间里抽样，帮我们用交叉验证自动找出表现最好的参数，让「选参数」从拍脑袋变成可复现的流程。

- 超参数在 fit 前指定（打个比方：超参数是“开机前就拧好的旋钮”，不是模型自己学着调的；网格搜索就是把这排旋钮的所有组合挨个拧一遍，看哪套表现最好。）
- 网格成本等于组合数乘折数
- 随机搜索适合连续或大空间
- 最终测试集只使用一次


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | 参见本节示例 | 先明确样本、特征、目标和验证方式，再训练模型。 | 搜索前反复查看测试集 |
| 模型、公式与诊断 | `grid.score()`、`pd.DataFrame()`、`.fit()`、`.sort_values()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 把预处理放在搜索外 |


## 例 1｜数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-112 -->
### 数学推导｜超参数搜索是在验证规则下求最优

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜每组参数都走同一交叉验证。** 对候选 $\theta$，计算 $\bar s(\theta)=\sum_ks_k(\theta)/K$。

**第 2 步｜在候选空间内选择。** $\theta^*=\arg\max_{\theta\in\Theta}\bar s(\theta)$。

**第 3 步｜用全部训练数据重拟合。** 选择完成后得到

$$
\hat f^*=Train(D_{train};\theta^*)
$$

再只在独立测试集评估一次。若用同一个测试集反复比较参数，它就已经变成验证集。

**把上面的关系收束为本章计算式：**

$$
\theta^*=\arg\max_{\theta\in\Theta}\frac{1}{K}\sum_{k=1}^{K}s_k(\theta)
$$

**符号解释：** $\Theta$ 是候选超参数空间，$s_k$ 是第 $k$ 折分数。

**代码对应：** 把预处理和模型放进 Pipeline，再用 GridSearchCV 或 RandomizedSearchCV。

**使用边界：** 搜索次数越多，验证集过拟合风险越高；最终仍需独立测试集。


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=101
)
pipe = make_pipeline(StandardScaler(), SVC())


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：修改一个数据字段或模型参数，观察验证与测试指标如何变化。

上一节的示例把 `StandardScaler()` 和 `SVC()` 放进流水线，并用 5 折交叉验证搜索 `svc__C` 与 `svc__gamma`。请在下方代码里做一件事：把 `train_test_split` 的 `random_state` 从 `101` 改成另一个值，或者把 `pipe` 换成不带 `StandardScaler()` 的模型，再重新运行，观察 `grid.best_score_` 与 `grid.score(X_test, y_test)` 是否改变，并用一句话说明原因。


In [ ]:
try:
    # 请在下方填写代码：修改一个数据字段或模型参数，观察指标变化。
    # 上一节已经构造了 X_train, X_test, y_train, y_test 与 pipe。
    from sklearn.model_selection import GridSearchCV

    # TODO: 复用上一节的变量，修改一个参数并运行一个小型网格搜索；
    #       记录 best_score_ 与测试集分数，并用一句话解释变化。
    change_note = (
        "待填写"  # 例如：把 random_state 改成 7，或去掉 StandardScaler
    )
    # TODO：请在下方完成 —— 练一练：修改一个数据字段或模型参数，观察验证与测试指标如何变化。 上一节的示例把 StandardScaler()
    # 和

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd

grid = GridSearchCV(
    pipe,
    {"svc__C": [0.1, 1, 10], "svc__gamma": ["scale", 0.01, 0.1]},
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
).fit(X_train, y_train)
print(
    "best:",
    grid.best_params_,
    "CV:",
    round(grid.best_score_, 3),
    "test:",
    round(grid.score(X_test, y_test), 3),
)
display(
    pd.DataFrame(grid.cv_results_)[
        [
            "param_svc__C",
            "param_svc__gamma",
            "mean_test_score",
            "std_test_score",
        ]
    ]
    .sort_values("mean_test_score", ascending=False)
    .head()
)


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 搜索前反复查看测试集
- 把预处理放在搜索外
- 搜索范围无业务或计算依据
- 只展示最佳分数不展示波动


## 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 112.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 112.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 112.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

使用 GridSearchCV 和 RandomizedSearchCV 在流水线内调参，并控制搜索空间和计算预算。


### 你已经掌握

- 正确命名流水线参数
- 运行网格搜索
- 运行随机搜索
- 读取 cv_results_ 与最佳模型


### 需要注意

- 搜索前反复查看测试集
- 把预处理放在搜索外
- 搜索范围无业务或计算依据
- 只展示最佳分数不展示波动


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# 完整示例：把带 StandardScaler 的流水线换成不缩放的 SVC，观察指标变化。
pipe_raw = SVC()
small_grid = GridSearchCV(
    pipe_raw,
    {"C": [0.1, 1], "gamma": ["scale", 0.01]},
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
).fit(X_train, y_train)

print("best params:", small_grid.best_params_)
print("CV best_score_:", round(small_grid.best_score_, 3))
print("test roc_auc:", round(small_grid.score(X_test, y_test), 3))


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
from scipy.stats import loguniform

random_search = RandomizedSearchCV(
    pipe,
    {"svc__C": loguniform(1e-2, 1e2), "svc__gamma": loguniform(1e-4, 1)},
    n_iter=10,
    cv=4,
    scoring="roc_auc",
    random_state=101,
    n_jobs=-1,
).fit(X_train, y_train)
print(random_search.best_params_, round(random_search.best_score_, 3))
